Rúbrica: 

A continuación se muestra la rúbrica con la que se va a corregir el examen:


| Apartado/Criterio | Ponderación | Nota |
| :-- | --- | --- |
| Ej. 1.1. Ha tratado de manera adecuada los datos de las columnas. | 1 | 1 |
| Ej. 1.1. Ha seguido un criterio adecuado para elegir las entradas del problema. | 1 | 1 |
| Ej. 1.2. Ha diseñado bien la red y el sistema de entrenamiento | 1 | 0,75 |
| Ej. 1.2. Ha dimensionado bien la red neuronal. | 1 | 1 |
| Ej. 1.2. Ha hecho modificaciones coherentes para conseguir un mejor resultado. | 0,5 | 0,3 |
| Ej. 1.2. Ha usado su experiencia para valorar si el resultado es válido o no. | 1 | 0 |
| Ej. 2.1. Ha cargado adecuadamente los datos. | 0,5 | 0,5 |
| Ej. 2.2. Ha diseñado bien la red y el sistema de entrenamiento | 1 | 0,75 |
| Ej. 2.2. Ha dimensionado bien la red neuronal. | 1 | 0,5 |
| Ej. 2.2. Ha hecho modificaciones coherentes para conseguir un mejor resultado. | 1 | 1 |
| Ej. 2.2. Ha usado su experiencia para valorar si el resultado es válido o no. | 1 | 0 |

TOTAL: 6,8



In [896]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tensorflow import keras
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVR
from sklearn.metrics import classification_report
import datetime
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import os

In [897]:
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

## Ejercicio 1

In [898]:
vuelos = pd.read_csv("/home/ciabd03/Descargas/vuelos_pakistan.csv")

vuelos.head()

,Flight_ID,Date,Month,Day_of_Week,Departure_City,Arrival_City,Route_Type,Aircraft_Type,Flight_Duration_Minutes,Passengers,...,Load_Factor_%,Ticket_Price_USD,Delay_Minutes,Delay_Category,On_Time_Status,Weather_Condition,Fuel_Consumption,CO2_Emissions,Customer_Rating,Customer_Feedback
0,PK2026_0001,2026-06-09,June,Tuesday,Jeddah,Islamabad,International,Airbus A320,83.0,120,...,66.67,1140.0,220,Severe,Delayed,Clear,6265l,15662.5kg,4.1,Dreadful customer support
1,PK2026_0002,2026-08-12,August,Wednesday,Dubai,Kuala Lumpur,International,Airbus A320,284.0,179,...,99.44,773.0,27,Minor,Delayed,NaN,3516l,8790.0kg,3.6,"Standard flight, nothing special"
2,PK2026_0003,2026-04-20,April,Monday,Doha,Lahore,International,ATR 72,333.0,69,...,98.57,155.0,176,Severe,Delayed,Fog,13538l,33845.0kg,3.0,Tardy arrival but very cozy
3,PK2026_0004,2026-12-07,December,Monday,Jeddah,Lahore,International,Boeing 777,330.0,291,...,83.14,1237.0,87,Moderate,Delayed,NaN,18850l,47125.0kg,NaN,Mediocre experience overall
4,PK2026_0005,2026-05-04,May,Monday,Lahore,Doha,International,Boeing 737,283.0,159,...,99.38,141.0,82,Moderate,Delayed,NaN,13474l,33685.0kg,3.0,Behind schedule but quite relaxed


In [899]:
vuelos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Flight_ID                800 non-null    object 
 1   Date                     800 non-null    object 
 2   Month                    800 non-null    object 
 3   Day_of_Week              800 non-null    object 
 4   Departure_City           800 non-null    object 
 5   Arrival_City             800 non-null    object 
 6   Route_Type               800 non-null    object 
 7   Aircraft_Type            800 non-null    object 
 8   Flight_Duration_Minutes  725 non-null    float64
 9   Passengers               800 non-null    int64  
 10  Seat_Capacity            800 non-null    int64  
 11  Load_Factor_%            800 non-null    float64
 12  Ticket_Price_USD         786 non-null    float64
 13  Delay_Minutes            800 non-null    int64  
 14  Delay_Category           8

In [900]:
vuelos = vuelos.drop(["Weather_Condition"],axis=1)

In [901]:
vuelos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Flight_ID                800 non-null    object 
 1   Date                     800 non-null    object 
 2   Month                    800 non-null    object 
 3   Day_of_Week              800 non-null    object 
 4   Departure_City           800 non-null    object 
 5   Arrival_City             800 non-null    object 
 6   Route_Type               800 non-null    object 
 7   Aircraft_Type            800 non-null    object 
 8   Flight_Duration_Minutes  725 non-null    float64
 9   Passengers               800 non-null    int64  
 10  Seat_Capacity            800 non-null    int64  
 11  Load_Factor_%            800 non-null    float64
 12  Ticket_Price_USD         786 non-null    float64
 13  Delay_Minutes            800 non-null    int64  
 14  Delay_Category           8

In [902]:
vuelos["Flight_Duration_Minutes"].fillna(vuelos["Flight_Duration_Minutes"].median(),inplace=True)

/tmp/ipykernel_25307/2744780676.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  vuelos["Flight_Duration_Minutes"].fillna(vuelos["Flight_Duration_Minutes"].median(),inplace=True)


In [903]:
vuelos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Flight_ID                800 non-null    object 
 1   Date                     800 non-null    object 
 2   Month                    800 non-null    object 
 3   Day_of_Week              800 non-null    object 
 4   Departure_City           800 non-null    object 
 5   Arrival_City             800 non-null    object 
 6   Route_Type               800 non-null    object 
 7   Aircraft_Type            800 non-null    object 
 8   Flight_Duration_Minutes  800 non-null    float64
 9   Passengers               800 non-null    int64  
 10  Seat_Capacity            800 non-null    int64  
 11  Load_Factor_%            800 non-null    float64
 12  Ticket_Price_USD         786 non-null    float64
 13  Delay_Minutes            800 non-null    int64  
 14  Delay_Category           8

In [ ]:
vuelos["Ticket_Price_USD"].fillna(vuelos["Ticket_Price_USD"].median(),inplace=True)
#CORRECCIÓN: El fillna con la columna objetivo no es buena idea por su relevancia. Nos estamos inventando resultados.

/tmp/ipykernel_25307/2753229801.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  vuelos["Ticket_Price_USD"].fillna(vuelos["Ticket_Price_USD"].median(),inplace=True)


In [905]:
vuelos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Flight_ID                800 non-null    object 
 1   Date                     800 non-null    object 
 2   Month                    800 non-null    object 
 3   Day_of_Week              800 non-null    object 
 4   Departure_City           800 non-null    object 
 5   Arrival_City             800 non-null    object 
 6   Route_Type               800 non-null    object 
 7   Aircraft_Type            800 non-null    object 
 8   Flight_Duration_Minutes  800 non-null    float64
 9   Passengers               800 non-null    int64  
 10  Seat_Capacity            800 non-null    int64  
 11  Load_Factor_%            800 non-null    float64
 12  Ticket_Price_USD         800 non-null    float64
 13  Delay_Minutes            800 non-null    int64  
 14  Delay_Category           8

In [906]:
vuelos["Customer_Rating"].fillna(vuelos["Customer_Rating"].median(),inplace=True)

/tmp/ipykernel_25307/1084780879.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  vuelos["Customer_Rating"].fillna(vuelos["Customer_Rating"].median(),inplace=True)


In [907]:
vuelos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Flight_ID                800 non-null    object 
 1   Date                     800 non-null    object 
 2   Month                    800 non-null    object 
 3   Day_of_Week              800 non-null    object 
 4   Departure_City           800 non-null    object 
 5   Arrival_City             800 non-null    object 
 6   Route_Type               800 non-null    object 
 7   Aircraft_Type            800 non-null    object 
 8   Flight_Duration_Minutes  800 non-null    float64
 9   Passengers               800 non-null    int64  
 10  Seat_Capacity            800 non-null    int64  
 11  Load_Factor_%            800 non-null    float64
 12  Ticket_Price_USD         800 non-null    float64
 13  Delay_Minutes            800 non-null    int64  
 14  Delay_Category           8

In [908]:
vuelos_clean = vuelos.copy()
vuelos_clean = vuelos_clean.drop(["Flight_ID","Date","Customer_Feedback"],axis=1)

In [909]:
vuelos_clean.head()

,Month,Day_of_Week,Departure_City,Arrival_City,Route_Type,Aircraft_Type,Flight_Duration_Minutes,Passengers,Seat_Capacity,Load_Factor_%,Ticket_Price_USD,Delay_Minutes,Delay_Category,On_Time_Status,Fuel_Consumption,CO2_Emissions,Customer_Rating
0,June,Tuesday,Jeddah,Islamabad,International,Airbus A320,83.0,120,180,66.67,1140.0,220,Severe,Delayed,6265l,15662.5kg,4.1
1,August,Wednesday,Dubai,Kuala Lumpur,International,Airbus A320,284.0,179,180,99.44,773.0,27,Minor,Delayed,3516l,8790.0kg,3.6
2,April,Monday,Doha,Lahore,International,ATR 72,333.0,69,70,98.57,155.0,176,Severe,Delayed,13538l,33845.0kg,3.0
3,December,Monday,Jeddah,Lahore,International,Boeing 777,330.0,291,350,83.14,1237.0,87,Moderate,Delayed,18850l,47125.0kg,3.7
4,May,Monday,Lahore,Doha,International,Boeing 737,283.0,159,160,99.38,141.0,82,Moderate,Delayed,13474l,33685.0kg,3.0


In [ ]:
Mes = []

for i, row in vuelos_clean.iterrows():
    if row["Month"] == "January":
        Mes.append(1)
    if row["Month"] == "February":
        Mes.append(2)
    if row["Month"] == "March":
        Mes.append(3)
    if row["Month"] == "April":
        Mes.append(4)
    if row["Month"] == "May":
        Mes.append(5)
    if row["Month"] == "June":
        Mes.append(6)
    if row["Month"] == "July":
        Mes.append(7)
    if row["Month"] == "August":
        Mes.append(8)
    if row["Month"] == "September":
        Mes.append(9)
    if row["Month"] == "October":
        Mes.append(10)
    if row["Month"] == "November":
        Mes.append(11)
    if row["Month"] == "December":
        Mes.append(12)

vuelos_clean["Month"] = Mes

#CORRECCIÓN: Ojo con el label encoding. En este caso, como en muchos, mete un pequeño sesgo.

In [911]:
vuelos_clean.head()
vuelos_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Month                    800 non-null    int64  
 1   Day_of_Week              800 non-null    object 
 2   Departure_City           800 non-null    object 
 3   Arrival_City             800 non-null    object 
 4   Route_Type               800 non-null    object 
 5   Aircraft_Type            800 non-null    object 
 6   Flight_Duration_Minutes  800 non-null    float64
 7   Passengers               800 non-null    int64  
 8   Seat_Capacity            800 non-null    int64  
 9   Load_Factor_%            800 non-null    float64
 10  Ticket_Price_USD         800 non-null    float64
 11  Delay_Minutes            800 non-null    int64  
 12  Delay_Category           800 non-null    object 
 13  On_Time_Status           800 non-null    object 
 14  Fuel_Consumption         8

In [912]:
Dia = []

for i, row in vuelos_clean.iterrows():
    if row["Day_of_Week"] == "Monday":
        Dia.append(1)
    if row["Day_of_Week"] == "Tuesday":
        Dia.append(2)
    if row["Day_of_Week"] == "Wednesday":
        Dia.append(3)
    if row["Day_of_Week"] == "Thursday":
        Dia.append(4)
    if row["Day_of_Week"] == "Friday":
        Dia.append(5)
    if row["Day_of_Week"] == "Saturday":
        Dia.append(6)
    if row["Day_of_Week"] == "Sunday":
        Dia.append(7)

vuelos_clean["Day_of_Week"] = Dia

In [913]:
vuelos_clean.head()
vuelos_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Month                    800 non-null    int64  
 1   Day_of_Week              800 non-null    int64  
 2   Departure_City           800 non-null    object 
 3   Arrival_City             800 non-null    object 
 4   Route_Type               800 non-null    object 
 5   Aircraft_Type            800 non-null    object 
 6   Flight_Duration_Minutes  800 non-null    float64
 7   Passengers               800 non-null    int64  
 8   Seat_Capacity            800 non-null    int64  
 9   Load_Factor_%            800 non-null    float64
 10  Ticket_Price_USD         800 non-null    float64
 11  Delay_Minutes            800 non-null    int64  
 12  Delay_Category           800 non-null    object 
 13  On_Time_Status           800 non-null    object 
 14  Fuel_Consumption         8

In [914]:
vuelos_clean.head()

,Month,Day_of_Week,Departure_City,Arrival_City,Route_Type,Aircraft_Type,Flight_Duration_Minutes,Passengers,Seat_Capacity,Load_Factor_%,Ticket_Price_USD,Delay_Minutes,Delay_Category,On_Time_Status,Fuel_Consumption,CO2_Emissions,Customer_Rating
0,6,2,Jeddah,Islamabad,International,Airbus A320,83.0,120,180,66.67,1140.0,220,Severe,Delayed,6265l,15662.5kg,4.1
1,8,3,Dubai,Kuala Lumpur,International,Airbus A320,284.0,179,180,99.44,773.0,27,Minor,Delayed,3516l,8790.0kg,3.6
2,4,1,Doha,Lahore,International,ATR 72,333.0,69,70,98.57,155.0,176,Severe,Delayed,13538l,33845.0kg,3.0
3,12,1,Jeddah,Lahore,International,Boeing 777,330.0,291,350,83.14,1237.0,87,Moderate,Delayed,18850l,47125.0kg,3.7
4,5,1,Lahore,Doha,International,Boeing 737,283.0,159,160,99.38,141.0,82,Moderate,Delayed,13474l,33685.0kg,3.0


In [915]:
co2 = []
fuel = []

for i, row in vuelos_clean.iterrows():
    co2.append(float(row["CO2_Emissions"].replace("kg","")))

for i, row in vuelos_clean.iterrows():
    fuel.append(float(row["Fuel_Consumption"].replace("l","")))

vuelos_clean["CO2_Emissions"] = co2
vuelos_clean["Fuel_Consumption"] = fuel

In [916]:
vuelos_clean.head()

,Month,Day_of_Week,Departure_City,Arrival_City,Route_Type,Aircraft_Type,Flight_Duration_Minutes,Passengers,Seat_Capacity,Load_Factor_%,Ticket_Price_USD,Delay_Minutes,Delay_Category,On_Time_Status,Fuel_Consumption,CO2_Emissions,Customer_Rating
0,6,2,Jeddah,Islamabad,International,Airbus A320,83.0,120,180,66.67,1140.0,220,Severe,Delayed,6265.0,15662.5,4.1
1,8,3,Dubai,Kuala Lumpur,International,Airbus A320,284.0,179,180,99.44,773.0,27,Minor,Delayed,3516.0,8790.0,3.6
2,4,1,Doha,Lahore,International,ATR 72,333.0,69,70,98.57,155.0,176,Severe,Delayed,13538.0,33845.0,3.0
3,12,1,Jeddah,Lahore,International,Boeing 777,330.0,291,350,83.14,1237.0,87,Moderate,Delayed,18850.0,47125.0,3.7
4,5,1,Lahore,Doha,International,Boeing 737,283.0,159,160,99.38,141.0,82,Moderate,Delayed,13474.0,33685.0,3.0


In [917]:
vuelos_clean = pd.get_dummies(vuelos_clean,dtype=int)

In [918]:
vuelos_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 39 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Month                        800 non-null    int64  
 1   Day_of_Week                  800 non-null    int64  
 2   Flight_Duration_Minutes      800 non-null    float64
 3   Passengers                   800 non-null    int64  
 4   Seat_Capacity                800 non-null    int64  
 5   Load_Factor_%                800 non-null    float64
 6   Ticket_Price_USD             800 non-null    float64
 7   Delay_Minutes                800 non-null    int64  
 8   Fuel_Consumption             800 non-null    float64
 9   CO2_Emissions                800 non-null    float64
 10  Customer_Rating              800 non-null    float64
 11  Departure_City_Doha          800 non-null    int64  
 12  Departure_City_Dubai         800 non-null    int64  
 13  Departure_City_Islam

In [919]:
vuelos_clean.corr()["Ticket_Price_USD"].abs().sort_values(ascending=False)[1:]

Arrival_City_Jeddah            0.078116
Departure_City_Karachi         0.067351
Month                          0.056245
Departure_City_Lahore          0.055104
Delay_Category_No Delay        0.051729
Departure_City_Dubai           0.045542
Arrival_City_Karachi           0.039336
Departure_City_Kuala Lumpur    0.036338
Delay_Category_Moderate        0.036223
Delay_Minutes                  0.032122
Arrival_City_London            0.030587
Passengers                     0.026216
Delay_Category_Severe          0.024043
Customer_Rating                0.023571
Arrival_City_Dubai             0.021898
On_Time_Status_Delayed         0.021497
On_Time_Status_On Time         0.021497
Aircraft_Type_Airbus A320      0.020245
Route_Type_International       0.019910
Route_Type_Domestic            0.019910
Arrival_City_Islamabad         0.019562
Arrival_City_Lahore            0.018860
CO2_Emissions                  0.018081
Fuel_Consumption               0.018081
Day_of_Week                    0.016370


In [920]:
X = vuelos_clean.drop(["Ticket_Price_USD"],axis=1)
y = vuelos_clean["Ticket_Price_USD"]

In [921]:
escalador = StandardScaler()
X = escalador.fit_transform(X)
y = escalador.fit_transform(y.to_frame())

In [922]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X,y,test_size=0.1,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full,y_train_full,test_size=0.1,random_state=42)

In [923]:
X_train.shape[1:]

(38,)

In [ ]:
model = keras.models.Sequential()
model.add(keras.layers.Dense(24,input_shape=X_train.shape[1:],activation="relu"))
model.add(keras.layers.Dense(5,activation="relu"))
model.add(keras.layers.Dense(1))

#CORRECCIÓN: El diseño de la red está bien (Acorde a regresión) y el dimensionamiento también está bien. Para el entrenamiento
# has puesto un early stopping aunque, con este tipo de ejercicio habría que dejar algo más de paciencia.
#No has hecho nada para valorar si el resultado está bien (un árbol de decisión o un random forest inicial).

/home/ciabd03/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [925]:
""" model = keras.models.Sequential()
model.add(keras.layers.Dense(24,input_shape=X_train.shape[1:],activation="relu"))
model.add(keras.layers.Dense(5,activation="relu"))
model.add(keras.layers.Dense(1, activation="relu")) """

' model = keras.models.Sequential()\nmodel.add(keras.layers.Dense(24,input_shape=X_train.shape[1:],activation="relu"))\nmodel.add(keras.layers.Dense(5,activation="relu"))\nmodel.add(keras.layers.Dense(1, activation="relu")) '

In [926]:
""" model = keras.models.Sequential()
model.add(keras.layers.Dense(30,input_shape=X_train.shape[1:],activation="relu"))
model.add(keras.layers.Dense(15,activation="relu"))
model.add(keras.layers.Dense(1)) """

' model = keras.models.Sequential()\nmodel.add(keras.layers.Dense(30,input_shape=X_train.shape[1:],activation="relu"))\nmodel.add(keras.layers.Dense(15,activation="relu"))\nmodel.add(keras.layers.Dense(1)) '

In [ ]:
""" model = keras.models.Sequential()
model.add(keras.layers.Dense(3,input_shape=X_train.shape[1:],activation="relu"))
model.add(keras.layers.Dense(2,activation="relu"))
model.add(keras.layers.Dense(1)) """
#CORRECCIÓN: Las pruebas que has hecho son coherentes pero tendrías que haber subido también el número de épocas.

' model = keras.models.Sequential()\nmodel.add(keras.layers.Dense(3,input_shape=X_train.shape[1:],activation="relu"))\nmodel.add(keras.layers.Dense(2,activation="relu"))\nmodel.add(keras.layers.Dense(1)) '

In [928]:
model.compile(loss="mean_squared_error", optimizer=keras.optimizers.SGD(learning_rate=0.0005), metrics=["mse"])

In [929]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True)

In [930]:
history = model.fit(X_train,y_train, epochs=1000,validation_data=(X_val,y_val),callbacks=[early_stopping_cb])

Epoch 1/1000
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.8989 - mse: 1.8989 - val_loss: 1.6673 - val_mse: 1.6673
Epoch 2/1000
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.7022 - mse: 1.7022 - val_loss: 1.5330 - val_mse: 1.5330
Epoch 3/1000
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.5581 - mse: 1.5581 - val_loss: 1.4335 - val_mse: 1.4335
Epoch 4/1000
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.4504 - mse: 1.4504 - val_loss: 1.3609 - val_mse: 1.3609
Epoch 5/1000
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.3701 - mse: 1.3701 - val_loss: 1.3065 - val_mse: 1.3065
Epoch 6/1000
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.3091 - mse: 1.3091 - val_loss: 1.2603 - val_mse: 1.2603
Epoch 7/1000
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.2569 - mse: 1.2569 - val_loss: 1.2276 - val_mse: 1.2276
Epoch 8/1000
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.2183 - mse: 1.2183 - val_loss: 1.1990 - val_mse: 1.1990
Epoch 9/1000
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - lo

In [931]:
model.evaluate(X_test,y_test)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1.2694 - mse: 1.2694 


[1.2693655490875244, 1.2693655490875244]

In [932]:
y_pred = model.predict(X_test)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


In [ ]:
r2_score(y_pred,y_test)
#CORRECCIÓN: Para poder valorar el resultado tendrías que haber comparado con un árbol de decisión o con un random forest.

-16.950468750428353

## Ejercicio 2

In [934]:
folders = listdir("/home/ciabd03/Descargas/cartas/")

photos = []
labels = []

for idx, folder in enumerate(folders):
    for file in listdir("/home/ciabd03/Descargas/cartas/"+folder):
        photo = load_img("/home/ciabd03/Descargas/cartas/"+folder+"/"+file, color_mode="grayscale",target_size=(117,117))
        photo = img_to_array(photo)
        photos.append(photo)
        labels.append(float(idx))
        del photo

    print(folder)
    print(idx)

six of clubs
0
three of clubs
1
jack of clubs
2
five of spades
3
jack of diamonds
4
three of spades
5
two of diamonds
6
jack of spades
7
two of clubs
8
two of hearts
9
ace of diamonds
10
jack of hearts
11
joker
12
eight of clubs
13
three of diamonds
14
king of clubs
15
seven of diamonds
16
ten of clubs
17
five of clubs
18
six of spades
19
queen of hearts
20
queen of spades
21
king of diamonds
22
ten of diamonds
23
seven of spades
24
queen of diamonds
25
ten of hearts
26
two of spades
27
seven of hearts
28
ace of clubs
29
three of hearts
30
five of diamonds
31
six of hearts
32
four of spades
33
king of hearts
34
ace of hearts
35
nine of spades
36
king of spades
37
four of hearts
38
ace of spades
39
five of hearts
40
four of clubs
41
nine of diamonds
42
nine of clubs
43
seven of clubs
44
ten of spades
45
six of diamonds
46
queen of clubs
47
eight of hearts
48
eight of diamonds
49
nine of hearts
50
eight of spades
51
four of diamonds
52


In [935]:
X = np.asarray(photos)
y = np.asarray(labels)

In [936]:
X.shape

(7624, 117, 117, 1)

In [937]:
X = X.reshape(7624,-1)

In [ ]:
X = X/255.0
#CORRECCIÓN: Carga de datos OK.

In [939]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X,y,test_size=0.1,random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full,y_train_full,test_size=0.1,random_state=42)

In [940]:
X_train.shape[1:]

(13689,)

In [941]:
""" model = keras.models.Sequential()
model.add(keras.layers.Flatten(input_shape=X_train.shape[1:]))
model.add(keras.layers.Dense(2000,activation="relu"))
model.add(keras.layers.Dense(500,activation="relu"))
model.add(keras.layers.Dense(150,activation="relu"))
model.add(keras.layers.Dense(53,activation="softmax")) """

' model = keras.models.Sequential()\nmodel.add(keras.layers.Flatten(input_shape=X_train.shape[1:]))\nmodel.add(keras.layers.Dense(2000,activation="relu"))\nmodel.add(keras.layers.Dense(500,activation="relu"))\nmodel.add(keras.layers.Dense(150,activation="relu"))\nmodel.add(keras.layers.Dense(53,activation="softmax")) '

In [ ]:
model = keras.models.Sequential()
model.add(keras.layers.Flatten(input_shape=X_train.shape[1:]))
model.add(keras.layers.Dense(1500,activation="relu"))
model.add(keras.layers.Dense(250,activation="relu"))
model.add(keras.layers.Dense(75,activation="relu"))
model.add(keras.layers.Dense(53,activation="softmax"))
#CORRECCIÓN: Ojo, en cuanto al diseño, todo bien, salvo que has hecho reshape y flatten. Cuidado con estas cosas... en este 
#caso no va a suponer un error pero en otros casos despistes similares pueden suponer problemas.
# En cuanto al dimensionamiento, 1500 es poco, no llega al 11%. De primeras habría que poner más.

' model = keras.models.Sequential()\nmodel.add(keras.layers.Flatten(input_shape=X_train.shape[1:]))\nmodel.add(keras.layers.Dense(1500,activation="relu"))\nmodel.add(keras.layers.Dense(250,activation="relu"))\nmodel.add(keras.layers.Dense(75,activation="relu"))\nmodel.add(keras.layers.Dense(53,activation="softmax")) '

In [ ]:
""" model = keras.models.Sequential()
model.add(keras.layers.Flatten(input_shape=X_train.shape[1:]))
model.add(keras.layers.Dense(2500,activation="relu"))
model.add(keras.layers.Dense(500,activation="relu"))
model.add(keras.layers.Dense(53,activation="softmax")) """

/home/ciabd03/anaconda3/lib/python3.13/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [944]:
model.compile(loss="sparse_categorical_crossentropy", optimizer=keras.optimizers.SGD(learning_rate=0.0005), metrics=["accuracy"])

In [945]:
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True)

In [946]:
history = model.fit(X_train,y_train, epochs=1000,validation_data=(X_val,y_val),callbacks=[early_stopping_cb])

Epoch 1/1000
193/193 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.0460 - loss: 3.9276 - val_accuracy: 0.0684 - val_loss: 3.8150
Epoch 2/1000
193/193 ━━━━━━━━━━━━━━━━━━━━ 9s 46ms/step - accuracy: 0.0973 - loss: 3.7369 - val_accuracy: 0.1310 - val_loss: 3.6623
Epoch 3/1000
193/193 ━━━━━━━━━━━━━━━━━━━━ 9s 46ms/step - accuracy: 0.1328 - loss: 3.6097 - val_accuracy: 0.1557 - val_loss: 3.5633
Epoch 4/1000
193/193 ━━━━━━━━━━━━━━━━━━━━ 9s 46ms/step - accuracy: 0.1582 - loss: 3.5016 - val_accuracy: 0.1528 - val_loss: 3.4866
Epoch 5/1000
193/193 ━━━━━━━━━━━━━━━━━━━━ 9s 47ms/step - accuracy: 0.1757 - loss: 3.4104 - val_accuracy: 0.1689 - val_loss: 3.4190
Epoch 6/1000
193/193 ━━━━━━━━━━━━━━━━━━━━ 9s 48ms/step - accuracy: 0.1931 - loss: 3.3280 - val_accuracy: 0.1689 - val_loss: 3.3669
Epoch 7/1000
193/193 ━━━━━━━━━━━━━━━━━━━━ 9s 49ms/step - accuracy: 0.2034 - loss: 3.2627 - val_accuracy: 0.2154 - val_loss: 3.2990
Epoch 8/1000
193/193 ━━━━━━━━━━━━━━━━━━━━ 9s 48ms/step - accuracy: 0.2178 - loss: 

KeyboardInterrupt: 

In [ ]:
model.evaluate(X_test,y_test)
#CORRECCIÓN: De los intentos que has hecho no veo los resultados. 
# No has usado métodos para valorar los resultados.

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3604 - loss: 2.5242


[2.5242297649383545, 0.3604193925857544]